In [0]:
%sql
-- Overall booking KPIs from the gold layer (same formulas as vw_annual_booking_kpis, all years combined)
SELECT
    YEAR(MIN(booking_date))                                 AS period_start,
    YEAR(MAX(booking_date))                                   AS period_end,
    SUM(ticket_price * (1 - discount))                    AS net_revenue,
    COUNT(DISTINCT booking_id)                             AS distinct_bookings,
    COUNT(DISTINCT passenger_id)                           AS active_passengers,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                       AS avg_booking_value
FROM tpca.gold.fact_bookings

In [0]:
%sql
-- Monthly net revenue trend across all years
SELECT
    DATE_FORMAT(booking_date, 'yyyy-MM')              AS booking_month,
    SUM(ticket_price * (1 - discount))                AS net_revenue,
    COUNT(DISTINCT booking_id)                         AS distinct_bookings,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                   AS avg_booking_value
FROM tpca.gold.fact_bookings
GROUP BY DATE_FORMAT(booking_date, 'yyyy-MM')
ORDER BY booking_month

In [0]:
%sql
-- Annual net revenue with year-over-year change
WITH annual AS (
    SELECT
        YEAR(booking_date)                        AS booking_year,
        SUM(ticket_price * (1 - discount))        AS net_revenue,
        COUNT(DISTINCT booking_id)                AS distinct_bookings
    FROM tpca.gold.fact_bookings
    GROUP BY YEAR(booking_date)
)
SELECT
    booking_year,
    ROUND(net_revenue, 2)                                                            AS net_revenue,
    distinct_bookings,
    ROUND(LAG(net_revenue) OVER (ORDER BY booking_year), 2)                          AS prev_year_revenue,
    ROUND(
        net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year),
        2
    )                                                                               AS yoy_change,
    ROUND(
        (net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year))
        / LAG(net_revenue) OVER (ORDER BY booking_year) * 100,
        2
    )                                                                               AS yoy_pct_change
FROM annual
ORDER BY booking_year

In [0]:
%sql
-- Net revenue by passenger travel purpose
SELECT
    travel_purpose,
    COUNT(DISTINCT passenger_id)                        AS passengers,
    COUNT(DISTINCT booking_id)                         AS distinct_bookings,
    SUM(ticket_price * (1 - discount))                AS net_revenue,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                   AS avg_booking_value,
    ROUND(
        SUM(ticket_price * (1 - discount)) * 100.0
        / SUM(SUM(ticket_price * (1 - discount))) OVER (),
        2
    )                                                   AS revenue_share_pct
FROM tpca.gold.fact_bookings
GROUP BY travel_purpose
ORDER BY net_revenue DESC

In [0]:
%sql
-- Top 10 passengers by net revenue
SELECT
    f.passenger_id,
    d.pa_name                                             AS passenger_name,
    COUNT(DISTINCT f.booking_id)                         AS distinct_bookings,
    SUM(f.seats)                                         AS total_seats,
    SUM(f.ticket_price * (1 - f.discount))              AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_passenger d
    ON f.passenger_id = d.pa_passengerid
GROUP BY f.passenger_id, d.pa_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Top 10 passengers as a share of total net revenue
WITH passenger_revenue AS (
    SELECT
        passenger_id,
        SUM(ticket_price * (1 - discount)) AS net_revenue
    FROM tpca.gold.fact_bookings
    GROUP BY passenger_id
),
ranked AS (
    SELECT
        passenger_id,
        net_revenue,
        ROUND(net_revenue * 100.0 / SUM(net_revenue) OVER (), 2) AS share_of_total_pct,
        SUM(net_revenue) OVER ()                                  AS grand_total,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC)         AS rnk
    FROM passenger_revenue
)
SELECT
    r.rnk,
    r.passenger_id,
    d.pa_name                                       AS passenger_name,
    ROUND(r.net_revenue, 2)                        AS net_revenue,
    r.share_of_total_pct,
    ROUND(
        SUM(r.net_revenue) OVER (ORDER BY r.rnk) * 100.0
        / MAX(r.grand_total) OVER (),
        2
    )                                               AS cumulative_share_pct
FROM ranked r
JOIN tpca.gold.dim_passenger d
    ON r.passenger_id = d.pa_passengerid
WHERE r.rnk <= 10
ORDER BY r.rnk

In [0]:
%sql
-- Top 10 routes by net revenue
SELECT
    f.route_id,
    d.ro_name                                              AS route_name,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.seats)                                          AS total_seats,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS share_of_total_pct
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_route d
    ON f.route_id = d.ro_routeid
GROUP BY f.route_id, d.ro_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Bottom 10 routes by net revenue
SELECT
    f.route_id,
    d.ro_name                                              AS route_name,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.seats)                                          AS total_seats,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS share_of_total_pct
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_route d
    ON f.route_id = d.ro_routeid
GROUP BY f.route_id, d.ro_name
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
%sql
-- Net revenue by route type
SELECT
    d.ro_routetype                                         AS route_type,
    COUNT(DISTINCT f.route_id)                            AS routes,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.seats)                                          AS total_seats,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS revenue_share_pct
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_route d
    ON f.route_id = d.ro_routeid
GROUP BY d.ro_routetype
ORDER BY net_revenue DESC

In [0]:
%sql
-- Top 10 airlines by net revenue
SELECT
    f.airline_id,
    d.al_name                                              AS airline_name,
    COUNT(DISTINCT f.route_id)                            AS routes_served,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.seats)                                          AS total_seats,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS share_of_total_pct
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_airline d
    ON f.airline_id = d.al_airlineid
GROUP BY f.airline_id, d.al_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Revenue share of the top 10 airlines
WITH airline_revenue AS (
    SELECT
        airline_id,
        SUM(ticket_price * (1 - discount)) AS net_revenue
    FROM tpca.gold.fact_bookings
    GROUP BY airline_id
),
ranked AS (
    SELECT
        airline_id,
        net_revenue,
        ROUND(net_revenue * 100.0 / SUM(net_revenue) OVER (), 2) AS share_of_total_pct,
        SUM(net_revenue) OVER ()                                  AS grand_total,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC)         AS rnk
    FROM airline_revenue
)
SELECT
    r.rnk,
    r.airline_id,
    d.al_name                                       AS airline_name,
    ROUND(r.net_revenue, 2)                        AS net_revenue,
    r.share_of_total_pct,
    ROUND(
        SUM(r.net_revenue) OVER (ORDER BY r.rnk) * 100.0
        / MAX(r.grand_total) OVER (),
        2
    )                                               AS cumulative_share_pct
FROM ranked r
JOIN tpca.gold.dim_airline d
    ON r.airline_id = d.al_airlineid
WHERE r.rnk <= 10
ORDER BY r.rnk

In [0]:
%sql
-- Net revenue by passenger region (via geography dim)
SELECT
    d.region_name                                         AS passenger_region,
    COUNT(DISTINCT f.nation)                              AS nations,
    COUNT(DISTINCT f.passenger_id)                        AS passengers,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS revenue_share_pct
FROM (
    SELECT
        passenger_id,
        passenger_nation_key                                AS nation,
        booking_id,
        ticket_price,
        discount
    FROM tpca.gold.fact_bookings
) f
JOIN tpca.gold.dim_geography d
    ON f.nation = d.n_nationkey
GROUP BY d.region_name
ORDER BY net_revenue DESC

In [0]:
%sql
-- Net revenue by passenger nation (all nations)
SELECT
    d.nation_name                                         AS passenger_nation,
    d.region_name                                         AS region,
    COUNT(DISTINCT f.passenger_id)                        AS passengers,
    COUNT(DISTINCT f.booking_id)                          AS distinct_bookings,
    SUM(f.ticket_price * (1 - f.discount))                AS net_revenue,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount))
        / COUNT(DISTINCT f.booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(f.ticket_price * (1 - f.discount)) * 100.0
        / SUM(SUM(f.ticket_price * (1 - f.discount))) OVER (),
        2
    )                                                       AS revenue_share_pct
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_geography d
    ON f.passenger_nation_key = d.n_nationkey
GROUP BY d.nation_name, d.region_name
ORDER BY net_revenue DESC

In [0]:
%sql
-- Revenue and volume split by refund flag
SELECT
    refund_flag,
    COUNT(DISTINCT booking_id)                          AS distinct_bookings,
    COUNT(*)                                            AS booking_lines,
    SUM(seats)                                          AS total_seats,
    SUM(ticket_price * (1 - discount))                AS net_revenue,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        SUM(ticket_price * (1 - discount)) * 100.0
        / SUM(SUM(ticket_price * (1 - discount))) OVER (),
        2
    )                                                       AS revenue_share_pct,
    ROUND(
        SUM(seats) * 100.0
        / SUM(SUM(seats)) OVER (),
        2
    )                                                       AS seat_share_pct
FROM tpca.gold.fact_bookings
GROUP BY refund_flag
ORDER BY net_revenue DESC

In [0]:
%sql
-- Average discount by travel purpose
SELECT
    travel_purpose,
    COUNT(DISTINCT booking_id)                          AS distinct_bookings,
    ROUND(AVG(discount), 4)                            AS avg_discount,
    ROUND(MIN(discount), 4)                            AS min_discount,
    ROUND(MAX(discount), 4)                            AS max_discount,
    ROUND(
        SUM(discount_amount) / NULLIF(SUM(gross_amount), 0),
        4
    )                                                       AS weighted_avg_discount,
    ROUND(
        SUM(discount_amount) * 100.0
        / NULLIF(SUM(gross_amount), 0),
        2
    )                                                       AS total_discount_leakage_pct
FROM tpca.gold.fact_bookings
GROUP BY travel_purpose
ORDER BY avg_discount DESC

In [0]:
%sql
-- Avg flight duration (departure to arrival, in days) by cabin class
SELECT
    cabin_class,
    COUNT(DISTINCT booking_id)                          AS distinct_bookings,
    COUNT(*)                                            AS booking_lines,
    AVG(DATEDIFF(arrival_date, departure_date))      AS avg_duration_days,
    MIN(DATEDIFF(arrival_date, departure_date))      AS min_duration_days,
    MAX(DATEDIFF(arrival_date, departure_date))      AS max_duration_days,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(*),
        2
    )                                                       AS avg_net_revenue_per_line
FROM tpca.gold.fact_bookings
GROUP BY cabin_class
ORDER BY avg_duration_days DESC

In [0]:
%sql
-- Booking count and revenue by booking priority
SELECT
    booking_priority,
    COUNT(DISTINCT booking_id)                          AS distinct_bookings,
    COUNT(*)                                            AS booking_lines,
    SUM(ticket_price * (1 - discount))                AS net_revenue,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                       AS avg_booking_value,
    ROUND(
        COUNT(DISTINCT booking_id) * 100.0
        / SUM(COUNT(DISTINCT booking_id)) OVER (),
        2
    )                                                       AS booking_share_pct,
    ROUND(
        SUM(ticket_price * (1 - discount)) * 100.0
        / SUM(SUM(ticket_price * (1 - discount))) OVER (),
        2
    )                                                       AS revenue_share_pct
FROM tpca.gold.fact_bookings
GROUP BY booking_priority
ORDER BY net_revenue DESC

In [0]:
%sql
-- Monthly breakdown of booking status
SELECT
    DATE_FORMAT(booking_date, 'yyyy-MM')                AS booking_month,
    booking_status,
    COUNT(DISTINCT booking_id)                          AS distinct_bookings,
    SUM(ticket_price * (1 - discount))                AS net_revenue,
    ROUND(
        COUNT(DISTINCT booking_id) * 100.0
        / SUM(COUNT(DISTINCT booking_id)) OVER (PARTITION BY DATE_FORMAT(booking_date, 'yyyy-MM')),
        2
    )                                                       AS bookings_share_of_month_pct
FROM tpca.gold.fact_bookings
GROUP BY DATE_FORMAT(booking_date, 'yyyy-MM'), booking_status
ORDER BY booking_month, booking_status

In [0]:
%sql
-- Route revenue ranking within each travel purpose (top 5 routes per purpose)
WITH route_purpose AS (
    SELECT
        travel_purpose,
        route_id,
        COUNT(DISTINCT booking_id)                    AS distinct_bookings,
        SUM(ticket_price * (1 - discount))            AS net_revenue
    FROM tpca.gold.fact_bookings
    GROUP BY travel_purpose, route_id
),
ranked AS (
    SELECT
        travel_purpose,
        route_id,
        distinct_bookings,
        net_revenue,
        SUM(net_revenue) OVER (PARTITION BY travel_purpose) AS purpose_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY travel_purpose
            ORDER BY net_revenue DESC
        )                                                   AS rnk
    FROM route_purpose
)
SELECT
    r.travel_purpose,
    r.rnk                                                   AS route_rank,
    d.ro_name                                             AS route_name,
    r.distinct_bookings,
    ROUND(r.net_revenue, 2)                              AS net_revenue,
    ROUND(
        r.net_revenue * 100.0 / r.purpose_revenue,
        2
    )                                                       AS share_of_purpose_pct
FROM ranked r
JOIN tpca.gold.dim_route d
    ON r.route_id = d.ro_routeid
WHERE r.rnk <= 5
ORDER BY r.travel_purpose, r.rnk

In [0]:
%sql
-- Revenue comparison: passenger home-region vs airline home-region
WITH passenger_geo AS (
    SELECT
        g.region_name                                       AS region,
        COUNT(DISTINCT f.booking_id)                       AS bookings,
        SUM(f.ticket_price * (1 - f.discount))          AS net_revenue
    FROM tpca.gold.fact_bookings f
    JOIN tpca.gold.dim_geography g
        ON f.passenger_nation_key = g.n_nationkey
    GROUP BY g.region_name
),
airline_geo AS (
    SELECT
        g.region_name                                       AS region,
        COUNT(DISTINCT f.booking_id)                       AS bookings,
        SUM(f.ticket_price * (1 - f.discount))          AS net_revenue
    FROM tpca.gold.fact_bookings f
    JOIN tpca.gold.dim_airline a
        ON f.airline_id = a.al_airlineid
    JOIN tpca.gold.dim_geography g
        ON a.al_nationkey = g.n_nationkey
    GROUP BY g.region_name
)
SELECT
    COALESCE(p.region, l.region)                          AS region,
    ROUND(p.net_revenue, 2)                               AS passenger_geo_revenue,
    ROUND(
        p.net_revenue * 100.0
        / SUM(p.net_revenue) OVER (),
        2
    )                                                       AS passenger_geo_share_pct,
    ROUND(l.net_revenue, 2)                               AS airline_geo_revenue,
    ROUND(
        l.net_revenue * 100.0
        / SUM(l.net_revenue) OVER (),
        2
    )                                                       AS airline_geo_share_pct,
    ROUND(
        l.net_revenue - p.net_revenue,
        2
    )                                                       AS revenue_gap
FROM passenger_geo p
FULL OUTER JOIN airline_geo l
    ON p.region = l.region
ORDER BY passenger_geo_revenue DESC